# Samsung Galaxy Watch Data Analysis

This notebook provides comprehensive analysis and visualization of Samsung Galaxy Watch health data stored in InfluxDB.

## Data Sources
- Heart rate monitoring
- Sleep analysis
- Activity tracking
- Stress monitoring
- Exercise sessions
- Advanced health metrics (HRV, SpO2, respiratory rate)

## Analysis Goals
1. Health trends over time
2. Sleep pattern analysis
3. Activity correlation with health metrics
4. Stress and wellness monitoring
5. Exercise performance analysis

## Setup and Configuration

In [25]:
# Import required libraries
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

# InfluxDB client
from influxdb_client import InfluxDBClient
from influxdb_client.client.query_api import QueryApi

# Configuration
from servers import INFLUXDB_URL, INFLUXDB_TOKEN, INFLUXDB_ORG, INFLUXDB_BUCKET

# Set up plotting style
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['font.size'] = 12

print("✓ Libraries imported successfully")
print(f"✓ InfluxDB URL: {INFLUXDB_URL}")
print(f"✓ InfluxDB Bucket: {INFLUXDB_BUCKET}")

✓ Libraries imported successfully
✓ InfluxDB URL: http://localhost:8086
✓ InfluxDB Bucket: watch_data


## Database Connection and Query Functions

In [26]:
# Initialize InfluxDB client
client = InfluxDBClient(url=INFLUXDB_URL, token=INFLUXDB_TOKEN, org=INFLUXDB_ORG)
query_api = client.query_api()

def query_measurement(measurement_name, start_time="-60d", fields=None, device_filter=None):
    """
    Query InfluxDB for a specific measurement
    
    Args:
        measurement_name: Name of the measurement (e.g., 'shealth.tracker.heart_rate')
        start_time: Start time for query (e.g., '-7d', '-1h')
        fields: List of fields to select (None for all)
        device_filter: Device UUID to filter by
    
    Returns:
        pandas.DataFrame: Query results
    """
    
    # Build field selection
    if fields:
        field_str = '|'.join([f'r._field == "{field}"' for field in fields])
        filter_clause = f'|> filter(fn: (r) => {field_str})'
    else:
        filter_clause = ''
    
    # Build device filter
    device_clause = f'|> filter(fn: (r) => r.deviceuuid == "{device_filter}")' if device_filter else ''
    
    query = f'''
    from(bucket: "{INFLUXDB_BUCKET}")
      |> range(start: {start_time})
      |> filter(fn: (r) => r._measurement == "{measurement_name}")
      {filter_clause}
      {device_clause}
      |> pivot(rowKey:["_time"], columnKey: ["_field"], valueColumn: "_value")
      |> sort(columns: ["_time"])
    '''
    
    try:
        result = query_api.query_data_frame(query)
        if not result.empty:
            result['_time'] = pd.to_datetime(result['_time'])
            result.set_index('_time', inplace=True)
        return result
    except Exception as e:
        print(f"Error querying {measurement_name}: {e}")
        return pd.DataFrame()

def get_available_measurements():
    """
    Get list of available measurements in the database
    """
    query = f'''
    import "influxdata/influxdb/schema"
    
    schema.measurements(bucket: "{INFLUXDB_BUCKET}")
    '''
    
    try:
        result = query_api.query_data_frame(query)
        return result['_value'].tolist() if not result.empty else []
    except Exception as e:
        print(f"Error getting measurements: {e}")
        return []

def get_device_list():
    """
    Get list of available devices in the database
    """
    query = f'''
    import "influxdata/influxdb/schema"
    
    schema.tagValues(
        bucket: "{INFLUXDB_BUCKET}",
        tag: "deviceuuid",
        start: -1y
    )
    '''
    
    try:
        result = query_api.query_data_frame(query)
        return result['_value'].tolist() if not result.empty else []
    except Exception as e:
        print(f"Error getting devices: {e}")
        return []

def get_device_measurement_counts():
    """
    Get measurement counts for each device to identify the most active device
    """
    query = f'''
    from(bucket: "{INFLUXDB_BUCKET}")
      |> range(start: -1y)
      |> group(columns: ["deviceuuid"])
      |> count()
      |> group()
      |> sort(columns: ["_value"], desc: true)
    '''
    
    try:
        result = query_api.query_data_frame(query)
        if not result.empty and 'deviceuuid' in result.columns:
            # Return list of tuples (device, count)
            return list(zip(result['deviceuuid'], result['_value']))
        return []
    except Exception as e:
        print(f"Error getting device measurement counts: {e}")
        return []

def get_primary_device():
    """
    Get the device with the most measurements for primary analysis
    """
    device_counts = get_device_measurement_counts()
    if device_counts:
        # Return device with highest count
        return device_counts[0][0]
    else:
        # Fallback to first device in list
        devices = get_device_list()
        return devices[0] if devices else None

print("✓ Database connection functions defined")

✓ Database connection functions defined


## Data Exploration

In [28]:
# Get available measurements and devices
measurements = get_available_measurements()
devices = get_device_list()

print("Available Measurements:")
for i, measurement in enumerate(measurements, 1):
    print(f"{i:2d}. {measurement}")

print(f"\nAvailable Devices ({len(devices)}):")
for i, device in enumerate(devices, 1):
    print(f"{i:2d}. {device}")

# Get device measurement counts to select the most active device
device_counts = get_device_measurement_counts()
if device_counts:
    print(f"\nDevice Measurement Counts:")
    for device, count in device_counts:
        print(f"  {device}: {count:,} measurements")

# Select primary device for analysis (device with most measurements)
primary_device = get_primary_device()
print(f"\nUsing primary device (most active): {primary_device}")

Available Measurements:
 1. health.advanced_glycation_endproduct.raw
 2. health.advanced_glycation_endproduct.raw_binned
 3. health.floors_climbed
 4. health.floors_climbed_binned
 5. health.movement
 6. health.movement_binned
 7. health.oxygen_saturation.raw
 8. health.oxygen_saturation.raw_binned
 9. health.respiratory_rate
10. health.respiratory_rate_binned
11. health.skin_temperature
12. health.skin_temperature_binned
13. shealth.activity.day_summary
14. shealth.activity.day_summary_binned
15. shealth.activity.day_summary_binned_hourlyDataList
16. shealth.activity.day_summary_binned_mActivityList
17. shealth.activity.day_summary_binned_mUnitDataList
18. shealth.activity_level
19. shealth.calories_burned.details
20. shealth.calories_burned.details_binned
21. shealth.calories_burned.details_binned_activityList
22. shealth.exercise
23. shealth.exercise_binned
24. shealth.tracker.pedometer_step_count
25. shealth.vitality_score

Available Devices (4):
 1. 64Z1605Zgy
 2. C8oz8zt6m4
 3. N

## 1. Heart Rate Analysis

In [16]:
# Query heart rate data
hr_data = query_measurement('shealth.tracker.heart_rate', start_time="-7", device_filter=primary_device)

if not hr_data.empty:
    print(f"Heart Rate Data: {len(hr_data)} records")
    print(f"Date range: {hr_data.index.min()} to {hr_data.index.max()}")
    print(f"Fields available: {[col for col in hr_data.columns if not col.startswith('_')]}")
    
    # Basic statistics
    if 'heart_rate' in hr_data.columns:
        hr_stats = hr_data['heart_rate'].describe()
        print(f"\nHeart Rate Statistics:")
        print(hr_stats)
        
        # Plot heart rate over time
        fig, axes = plt.subplots(2, 2, figsize=(15, 10))
        
        # Time series plot
        axes[0, 0].plot(hr_data.index, hr_data['heart_rate'], alpha=0.7)
        axes[0, 0].set_title('Heart Rate Over Time')
        axes[0, 0].set_ylabel('Heart Rate (BPM)')
        axes[0, 0].tick_params(axis='x', rotation=45)
        
        # Heart rate distribution
        axes[0, 1].hist(hr_data['heart_rate'].dropna(), bins=30, alpha=0.7, edgecolor='black')
        axes[0, 1].set_title('Heart Rate Distribution')
        axes[0, 1].set_xlabel('Heart Rate (BPM)')
        axes[0, 1].set_ylabel('Frequency')
        
        # Daily pattern (if enough data)
        hr_data['hour'] = hr_data.index.hour
        hourly_avg = hr_data.groupby('hour')['heart_rate'].mean()
        axes[1, 0].plot(hourly_avg.index, hourly_avg.values, marker='o')
        axes[1, 0].set_title('Average Heart Rate by Hour')
        axes[1, 0].set_xlabel('Hour of Day')
        axes[1, 0].set_ylabel('Average Heart Rate (BPM)')
        axes[1, 0].set_xticks(range(0, 24, 2))
        
        # Box plot by day
        hr_data['day'] = hr_data.index.date
        recent_days = hr_data.groupby('day')['heart_rate'].apply(list).tail(7)
        axes[1, 1].boxplot(recent_days.values, labels=[str(d) for d in recent_days.index])
        axes[1, 1].set_title('Heart Rate Distribution by Day')
        axes[1, 1].set_ylabel('Heart Rate (BPM)')
        axes[1, 1].tick_params(axis='x', rotation=45)
        
        plt.tight_layout()
        plt.show()
else:
    print("No heart rate data available")

No heart rate data available


## 2. Heart Rate Variability (HRV) Analysis

In [17]:
# Query HRV data (from binned data)
hrv_data = query_measurement('health.hrv_binned', start_time="-7d", device_filter=primary_device)

if not hrv_data.empty:
    print(f"HRV Data: {len(hrv_data)} records")
    print(f"Fields available: {[col for col in hrv_data.columns if not col.startswith('_')]}")
    
    fig, axes = plt.subplots(2, 2, figsize=(15, 10))
    
    # SDNN over time
    if 'sdnn' in hrv_data.columns:
        axes[0, 0].plot(hrv_data.index, hrv_data['sdnn'], marker='o', alpha=0.7)
        axes[0, 0].set_title('SDNN (Standard Deviation of NN Intervals)')
        axes[0, 0].set_ylabel('SDNN (ms)')
        axes[0, 0].tick_params(axis='x', rotation=45)
    
    # RMSSD over time
    if 'rmssd' in hrv_data.columns:
        axes[0, 1].plot(hrv_data.index, hrv_data['rmssd'], marker='o', alpha=0.7, color='orange')
        axes[0, 1].set_title('RMSSD (Root Mean Square of Successive Differences)')
        axes[0, 1].set_ylabel('RMSSD (ms)')
        axes[0, 1].tick_params(axis='x', rotation=45)
    
    # HRV correlation
    if 'sdnn' in hrv_data.columns and 'rmssd' in hrv_data.columns:
        axes[1, 0].scatter(hrv_data['sdnn'], hrv_data['rmssd'], alpha=0.7)
        axes[1, 0].set_title('SDNN vs RMSSD Correlation')
        axes[1, 0].set_xlabel('SDNN (ms)')
        axes[1, 0].set_ylabel('RMSSD (ms)')
        
        # Calculate correlation
        correlation = hrv_data[['sdnn', 'rmssd']].corr().iloc[0, 1]
        axes[1, 0].text(0.05, 0.95, f'r = {correlation:.3f}', transform=axes[1, 0].transAxes, 
                       verticalalignment='top', bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
    
    # HRV distribution
    if 'sdnn' in hrv_data.columns and 'rmssd' in hrv_data.columns:
        axes[1, 1].hist(hrv_data['sdnn'].dropna(), bins=20, alpha=0.7, label='SDNN', edgecolor='black')
        axes[1, 1].hist(hrv_data['rmssd'].dropna(), bins=20, alpha=0.7, label='RMSSD', edgecolor='black')
        axes[1, 1].set_title('HRV Metrics Distribution')
        axes[1, 1].set_xlabel('Value (ms)')
        axes[1, 1].set_ylabel('Frequency')
        axes[1, 1].legend()
    
    plt.tight_layout()
    plt.show()
    
    # HRV summary statistics
    if 'sdnn' in hrv_data.columns and 'rmssd' in hrv_data.columns:
        print("\nHRV Summary Statistics:")
        print(hrv_data[['sdnn', 'rmssd']].describe())
else:
    print("No HRV data available")

No HRV data available


## 3. Sleep Analysis

In [18]:
# Query sleep data
sleep_data = query_measurement('shealth.sleep', start_time="-14d", device_filter=primary_device)

if not sleep_data.empty:
    print(f"Sleep Data: {len(sleep_data)} records")
    print(f"Fields available: {[col for col in sleep_data.columns if not col.startswith('_')]}")
    
    # Sleep metrics analysis
    sleep_metrics = ['sleep_score', 'efficiency', 'sleep_duration', 'total_deep_duration', 'total_rem_duration', 'total_light_duration']
    available_metrics = [col for col in sleep_metrics if col in sleep_data.columns]
    
    if available_metrics:
        fig, axes = plt.subplots(2, 2, figsize=(15, 10))
        
        # Sleep score over time
        if 'sleep_score' in sleep_data.columns:
            axes[0, 0].plot(sleep_data.index, sleep_data['sleep_score'], marker='o', alpha=0.7)
            axes[0, 0].set_title('Sleep Score Over Time')
            axes[0, 0].set_ylabel('Sleep Score')
            axes[0, 0].tick_params(axis='x', rotation=45)
        
        # Sleep efficiency
        if 'efficiency' in sleep_data.columns:
            axes[0, 1].plot(sleep_data.index, sleep_data['efficiency'], marker='o', alpha=0.7, color='green')
            axes[0, 1].set_title('Sleep Efficiency Over Time')
            axes[0, 1].set_ylabel('Efficiency (%)')
            axes[0, 1].tick_params(axis='x', rotation=45)
        
        # Sleep duration breakdown
        duration_cols = ['total_deep_duration', 'total_rem_duration', 'total_light_duration']
        available_durations = [col for col in duration_cols if col in sleep_data.columns]
        
        if available_durations:
            sleep_avg = sleep_data[available_durations].mean()
            axes[1, 0].pie(sleep_avg.values, labels=['Deep', 'REM', 'Light'], autopct='%1.1f%%')
            axes[1, 0].set_title('Average Sleep Stage Distribution')
        
        # Sleep duration vs score correlation
        if 'sleep_duration' in sleep_data.columns and 'sleep_score' in sleep_data.columns:
            axes[1, 1].scatter(sleep_data['sleep_duration'], sleep_data['sleep_score'], alpha=0.7)
            axes[1, 1].set_title('Sleep Duration vs Sleep Score')
            axes[1, 1].set_xlabel('Sleep Duration (minutes)')
            axes[1, 1].set_ylabel('Sleep Score')
            
            # Add correlation
            corr = sleep_data[['sleep_duration', 'sleep_score']].corr().iloc[0, 1]
            axes[1, 1].text(0.05, 0.95, f'r = {corr:.3f}', transform=axes[1, 1].transAxes, 
                           verticalalignment='top', bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
        
        plt.tight_layout()
        plt.show()
        
        # Sleep statistics
        print("\nSleep Statistics:")
        print(sleep_data[available_metrics].describe())
    
    # Sleep stage analysis
    sleep_stages = query_measurement('health.sleep_stage', start_time="-7d", device_filter=primary_device)
    
    if not sleep_stages.empty and 'stage' in sleep_stages.columns:
        print("\nSleep Stage Analysis:")
        stage_mapping = {40001: 'Light', 40002: 'Deep', 40003: 'REM', 40004: 'Awake'}
        
        # Map stage codes to names
        sleep_stages['stage_name'] = sleep_stages['stage'].map(stage_mapping)
        stage_counts = sleep_stages['stage_name'].value_counts()
        
        plt.figure(figsize=(10, 6))
        stage_counts.plot(kind='bar', color=['lightblue', 'darkblue', 'purple', 'red'])
        plt.title('Sleep Stage Distribution (Last 7 Days)')
        plt.xlabel('Sleep Stage')
        plt.ylabel('Frequency')
        plt.xticks(rotation=45)
        plt.tight_layout()
        plt.show()
        
        print(stage_counts)

else:
    print("No sleep data available")

No sleep data available


## 4. Activity and Exercise Analysis

In [19]:
# Query step count data
steps_data = query_measurement('shealth.tracker.pedometer_step_count', start_time="-7d", device_filter=primary_device)

if not steps_data.empty:
    print(f"Steps Data: {len(steps_data)} records")
    print(f"Fields available: {[col for col in steps_data.columns if not col.startswith('_')]}")
    
    if 'count' in steps_data.columns:
        # Aggregate daily steps
        steps_data['date'] = steps_data.index.date
        daily_steps = steps_data.groupby('date').agg({
            'count': 'sum',
            'distance': 'sum',
            'calorie': 'sum'
        })
        
        fig, axes = plt.subplots(2, 2, figsize=(15, 10))
        
        # Daily steps
        axes[0, 0].bar(range(len(daily_steps)), daily_steps['count'], alpha=0.7)
        axes[0, 0].set_title('Daily Step Count')
        axes[0, 0].set_ylabel('Steps')
        axes[0, 0].set_xticks(range(len(daily_steps)))
        axes[0, 0].set_xticklabels([str(d) for d in daily_steps.index], rotation=45)
        
        # Hourly pattern
        steps_data['hour'] = steps_data.index.hour
        hourly_steps = steps_data.groupby('hour')['count'].sum()
        axes[0, 1].plot(hourly_steps.index, hourly_steps.values, marker='o')
        axes[0, 1].set_title('Hourly Step Pattern')
        axes[0, 1].set_xlabel('Hour of Day')
        axes[0, 1].set_ylabel('Total Steps')
        axes[0, 1].set_xticks(range(0, 24, 2))
        
        # Distance vs steps correlation
        if 'distance' in steps_data.columns:
            axes[1, 0].scatter(steps_data['count'], steps_data['distance'], alpha=0.5)
            axes[1, 0].set_title('Steps vs Distance')
            axes[1, 0].set_xlabel('Steps')
            axes[1, 0].set_ylabel('Distance (m)')
        
        # Calories vs steps
        if 'calorie' in steps_data.columns:
            axes[1, 1].scatter(steps_data['count'], steps_data['calorie'], alpha=0.5, color='red')
            axes[1, 1].set_title('Steps vs Calories')
            axes[1, 1].set_xlabel('Steps')
            axes[1, 1].set_ylabel('Calories')
        
        plt.tight_layout()
        plt.show()
        
        print("\nDaily Activity Summary:")
        print(daily_steps)
        
        print(f"\nAverage daily steps: {daily_steps['count'].mean():.0f}")
        print(f"Average daily distance: {daily_steps['distance'].mean():.0f} meters")
        print(f"Average daily calories: {daily_steps['calorie'].mean():.1f} kcal")

# Query exercise data
exercise_data = query_measurement('shealth.exercise', start_time="-14d", device_filter=primary_device)

if not exercise_data.empty:
    print(f"\nExercise Data: {len(exercise_data)} records")
    print(f"Fields available: {[col for col in exercise_data.columns if not col.startswith('_')]}")
    
    # Exercise statistics
    exercise_metrics = ['duration', 'distance', 'calorie', 'mean_heart_rate', 'max_heart_rate']
    available_exercise = [col for col in exercise_metrics if col in exercise_data.columns]
    
    if available_exercise:
        print("\nExercise Statistics:")
        print(exercise_data[available_exercise].describe())
        
        # Exercise performance over time
        if len(exercise_data) > 1:
            fig, axes = plt.subplots(2, 2, figsize=(15, 10))
            
            if 'duration' in exercise_data.columns:
                axes[0, 0].plot(exercise_data.index, exercise_data['duration']/60000, marker='o')  # Convert to minutes
                axes[0, 0].set_title('Exercise Duration Over Time')
                axes[0, 0].set_ylabel('Duration (minutes)')
                axes[0, 0].tick_params(axis='x', rotation=45)
            
            if 'calorie' in exercise_data.columns:
                axes[0, 1].plot(exercise_data.index, exercise_data['calorie'], marker='o', color='red')
                axes[0, 1].set_title('Calories Burned Per Exercise')
                axes[0, 1].set_ylabel('Calories')
                axes[0, 1].tick_params(axis='x', rotation=45)
            
            if 'mean_heart_rate' in exercise_data.columns:
                axes[1, 0].plot(exercise_data.index, exercise_data['mean_heart_rate'], marker='o', color='green')
                axes[1, 0].set_title('Mean Heart Rate During Exercise')
                axes[1, 0].set_ylabel('Heart Rate (BPM)')
                axes[1, 0].tick_params(axis='x', rotation=45)
            
            if 'distance' in exercise_data.columns and 'duration' in exercise_data.columns:
                # Calculate speed (distance/time)
                speed = (exercise_data['distance'] / (exercise_data['duration'] / 1000)) * 3.6  # km/h
                axes[1, 1].plot(exercise_data.index, speed, marker='o', color='purple')
                axes[1, 1].set_title('Exercise Speed Over Time')
                axes[1, 1].set_ylabel('Speed (km/h)')
                axes[1, 1].tick_params(axis='x', rotation=45)
            
            plt.tight_layout()
            plt.show()

else:
    print("No activity data available")

No activity data available


## 5. Stress Monitoring

In [20]:
# Query stress data
stress_data = query_measurement('shealth.stress', start_time="-7d", device_filter=primary_device)

if not stress_data.empty:
    print(f"Stress Data: {len(stress_data)} records")
    print(f"Fields available: {[col for col in stress_data.columns if not col.startswith('_')]}")
    
    # Also query binned stress data
    stress_binned = query_measurement('shealth.stress_binned', start_time="-7d", device_filter=primary_device)
    
    if not stress_binned.empty:
        print(f"Stress Binned Data: {len(stress_binned)} records")
        
        fig, axes = plt.subplots(2, 2, figsize=(15, 10))
        
        # Stress score over time
        if 'score' in stress_binned.columns:
            axes[0, 0].plot(stress_binned.index, stress_binned['score'], alpha=0.7)
            axes[0, 0].set_title('Stress Score Over Time')
            axes[0, 0].set_ylabel('Stress Score')
            axes[0, 0].tick_params(axis='x', rotation=45)
        
        # Stress level distribution
        if 'level' in stress_binned.columns:
            level_counts = stress_binned['level'].value_counts().sort_index()
            level_names = {1: 'Low', 2: 'Moderate', 3: 'High', 4: 'Very High'}
            
            axes[0, 1].bar(range(len(level_counts)), level_counts.values, alpha=0.7)
            axes[0, 1].set_title('Stress Level Distribution')
            axes[0, 1].set_ylabel('Frequency')
            axes[0, 1].set_xticks(range(len(level_counts)))
            axes[0, 1].set_xticklabels([level_names.get(i, str(i)) for i in level_counts.index])
        
        # Daily stress pattern
        if 'score' in stress_binned.columns:
            stress_binned['hour'] = stress_binned.index.hour
            hourly_stress = stress_binned.groupby('hour')['score'].mean()
            axes[1, 0].plot(hourly_stress.index, hourly_stress.values, marker='o')
            axes[1, 0].set_title('Average Stress by Hour')
            axes[1, 0].set_xlabel('Hour of Day')
            axes[1, 0].set_ylabel('Average Stress Score')
            axes[1, 0].set_xticks(range(0, 24, 2))
        
        # Stress score vs level correlation
        if 'score' in stress_binned.columns and 'level' in stress_binned.columns:
            axes[1, 1].scatter(stress_binned['score'], stress_binned['level'], alpha=0.5)
            axes[1, 1].set_title('Stress Score vs Level')
            axes[1, 1].set_xlabel('Stress Score')
            axes[1, 1].set_ylabel('Stress Level')
        
        plt.tight_layout()
        plt.show()
        
        # Stress statistics
        stress_metrics = ['score', 'score_max', 'score_min', 'level']
        available_stress = [col for col in stress_metrics if col in stress_binned.columns]
        
        if available_stress:
            print("\nStress Statistics:")
            print(stress_binned[available_stress].describe())
            
            print(f"\nAverage stress score: {stress_binned['score'].mean():.1f}")
            print(f"Peak stress score: {stress_binned['score'].max():.1f}")
            print(f"Most common stress level: {stress_binned['level'].mode().iloc[0]}")

else:
    print("No stress data available")

No stress data available


## 6. Advanced Health Metrics

In [21]:
# Query oxygen saturation data
spo2_data = query_measurement('health.oxygen_saturation.raw_binned', start_time="-7d", device_filter=primary_device)

if not spo2_data.empty:
    print(f"SpO2 Data: {len(spo2_data)} records")
    print(f"Fields available: {[col for col in spo2_data.columns if not col.startswith('_')]}")
    
    # Note: SpO2 data might be in raw sensor format, need to check actual values
    print("\nSpO2 Data Sample:")
    print(spo2_data.head())

# Query skin temperature data
temp_data = query_measurement('health.skin_temperature', start_time="-7d", device_filter=primary_device)

if not temp_data.empty:
    print(f"\nSkin Temperature Data: {len(temp_data)} records")
    print(f"Fields available: {[col for col in temp_data.columns if not col.startswith('_')]}")
    
    if 'temperature' in temp_data.columns:
        fig, axes = plt.subplots(2, 2, figsize=(15, 10))
        
        # Temperature over time
        axes[0, 0].plot(temp_data.index, temp_data['temperature'], alpha=0.7)
        axes[0, 0].set_title('Skin Temperature Over Time')
        axes[0, 0].set_ylabel('Temperature (°C)')
        axes[0, 0].tick_params(axis='x', rotation=45)
        
        # Temperature distribution
        axes[0, 1].hist(temp_data['temperature'].dropna(), bins=30, alpha=0.7, edgecolor='black')
        axes[0, 1].set_title('Temperature Distribution')
        axes[0, 1].set_xlabel('Temperature (°C)')
        axes[0, 1].set_ylabel('Frequency')
        
        # Daily temperature pattern
        temp_data['hour'] = temp_data.index.hour
        hourly_temp = temp_data.groupby('hour')['temperature'].mean()
        axes[1, 0].plot(hourly_temp.index, hourly_temp.values, marker='o')
        axes[1, 0].set_title('Average Temperature by Hour')
        axes[1, 0].set_xlabel('Hour of Day')
        axes[1, 0].set_ylabel('Temperature (°C)')
        axes[1, 0].set_xticks(range(0, 24, 2))
        
        # Temperature vs baseline
        if 'baseline' in temp_data.columns:
            axes[1, 1].scatter(temp_data['baseline'], temp_data['temperature'], alpha=0.5)
            axes[1, 1].set_title('Temperature vs Baseline')
            axes[1, 1].set_xlabel('Baseline Temperature (°C)')
            axes[1, 1].set_ylabel('Measured Temperature (°C)')
            
            # Add perfect correlation line
            min_temp = min(temp_data['baseline'].min(), temp_data['temperature'].min())
            max_temp = max(temp_data['baseline'].max(), temp_data['temperature'].max())
            axes[1, 1].plot([min_temp, max_temp], [min_temp, max_temp], 'r--', alpha=0.5)
        
        plt.tight_layout()
        plt.show()
        
        print("\nTemperature Statistics:")
        print(temp_data[['temperature', 'baseline']].describe())

# Query respiratory rate data
resp_data = query_measurement('health.respiratory_rate', start_time="-7d", device_filter=primary_device)

if not resp_data.empty:
    print(f"\nRespiratory Rate Data: {len(resp_data)} records")
    print(f"Fields available: {[col for col in resp_data.columns if not col.startswith('_')]}")
    
    if 'average' in resp_data.columns:
        plt.figure(figsize=(12, 6))
        plt.plot(resp_data.index, resp_data['average'], marker='o', alpha=0.7)
        
        # Add confidence intervals if available
        if 'lower_limit' in resp_data.columns and 'upper_limit' in resp_data.columns:
            plt.fill_between(resp_data.index, resp_data['lower_limit'], resp_data['upper_limit'], alpha=0.2)
        
        plt.title('Respiratory Rate Over Time')
        plt.ylabel('Respiratory Rate (breaths/min)')
        plt.xticks(rotation=45)
        plt.tight_layout()
        plt.show()
        
        print("\nRespiratory Rate Statistics:")
        print(resp_data[['average', 'lower_limit', 'upper_limit']].describe())

if spo2_data.empty and temp_data.empty and resp_data.empty:
    print("No advanced health metrics data available")

No advanced health metrics data available


## 7. Multi-Metric Correlation Analysis

In [22]:
# Combine multiple metrics for correlation analysis
print("Building comprehensive health dashboard...")

# Collect all available daily metrics
daily_metrics = pd.DataFrame()

# Heart rate metrics
if not hr_data.empty and 'heart_rate' in hr_data.columns:
    hr_daily = hr_data.groupby(hr_data.index.date).agg({
        'heart_rate': ['mean', 'min', 'max', 'std']
    }).round(2)
    hr_daily.columns = ['hr_mean', 'hr_min', 'hr_max', 'hr_std']
    daily_metrics = pd.concat([daily_metrics, hr_daily], axis=1)

# Sleep metrics
if not sleep_data.empty:
    sleep_daily = sleep_data.groupby(sleep_data.index.date).agg({
        col: 'mean' for col in ['sleep_score', 'efficiency', 'sleep_duration'] if col in sleep_data.columns
    }).round(2)
    daily_metrics = pd.concat([daily_metrics, sleep_daily], axis=1)

# Activity metrics
if not steps_data.empty and 'count' in steps_data.columns:
    activity_daily = steps_data.groupby(steps_data.index.date).agg({
        'count': 'sum',
        'distance': 'sum',
        'calorie': 'sum'
    }).round(2)
    activity_daily.columns = ['daily_steps', 'daily_distance', 'daily_calories']
    daily_metrics = pd.concat([daily_metrics, activity_daily], axis=1)

# Stress metrics
if not stress_data.empty:
    stress_daily = stress_data.groupby(stress_data.index.date).agg({
        col: 'mean' for col in ['score', 'max', 'min'] if col in stress_data.columns
    }).round(2)
    stress_daily.columns = ['stress_avg', 'stress_max', 'stress_min']
    daily_metrics = pd.concat([daily_metrics, stress_daily], axis=1)

if not daily_metrics.empty:
    print(f"\nDaily Metrics Summary ({len(daily_metrics)} days):")
    print(daily_metrics.head())
    
    # Correlation matrix
    correlation_matrix = daily_metrics.corr()
    
    # Plot correlation heatmap
    plt.figure(figsize=(12, 10))
    mask = np.triu(np.ones_like(correlation_matrix, dtype=bool))
    sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', center=0,
                square=True, mask=mask, fmt='.2f')
    plt.title('Health Metrics Correlation Matrix')
    plt.tight_layout()
    plt.show()
    
    # Find strongest correlations
    print("\nStrongest Correlations:")
    
    # Get correlation pairs (excluding self-correlation)
    corr_pairs = []
    for i in range(len(correlation_matrix.columns)):
        for j in range(i+1, len(correlation_matrix.columns)):
            col1 = correlation_matrix.columns[i]
            col2 = correlation_matrix.columns[j]
            corr_val = correlation_matrix.iloc[i, j]
            if not np.isnan(corr_val):
                corr_pairs.append((abs(corr_val), corr_val, col1, col2))
    
    # Sort by absolute correlation value
    corr_pairs.sort(reverse=True)
    
    for abs_corr, corr, col1, col2 in corr_pairs[:10]:  # Top 10 correlations
        print(f"{col1} ↔ {col2}: {corr:.3f}")
    
    # Health trends over time
    if len(daily_metrics) > 3:
        fig, axes = plt.subplots(2, 2, figsize=(15, 10))
        
        # Multi-metric trend
        key_metrics = ['hr_mean', 'sleep_score', 'daily_steps', 'stress_avg']
        available_key = [col for col in key_metrics if col in daily_metrics.columns]
        
        if available_key:
            # Normalize metrics to 0-1 scale for comparison
            normalized_metrics = daily_metrics[available_key].copy()
            for col in normalized_metrics.columns:
                col_min = normalized_metrics[col].min()
                col_max = normalized_metrics[col].max()
                if col_max != col_min:
                    normalized_metrics[col] = (normalized_metrics[col] - col_min) / (col_max - col_min)
            
            axes[0, 0].plot(normalized_metrics.index, normalized_metrics, marker='o', alpha=0.7)
            axes[0, 0].set_title('Normalized Health Metrics Trends')
            axes[0, 0].set_ylabel('Normalized Value (0-1)')
            axes[0, 0].legend(available_key)
            axes[0, 0].tick_params(axis='x', rotation=45)
        
        # Heart rate vs sleep quality
        if 'hr_mean' in daily_metrics.columns and 'sleep_score' in daily_metrics.columns:
            axes[0, 1].scatter(daily_metrics['hr_mean'], daily_metrics['sleep_score'], alpha=0.7)
            axes[0, 1].set_title('Heart Rate vs Sleep Quality')
            axes[0, 1].set_xlabel('Average Heart Rate (BPM)')
            axes[0, 1].set_ylabel('Sleep Score')
        
        # Activity vs sleep
        if 'daily_steps' in daily_metrics.columns and 'sleep_duration' in daily_metrics.columns:
            axes[1, 0].scatter(daily_metrics['daily_steps'], daily_metrics['sleep_duration'], alpha=0.7)
            axes[1, 0].set_title('Daily Steps vs Sleep Duration')
            axes[1, 0].set_xlabel('Daily Steps')
            axes[1, 0].set_ylabel('Sleep Duration (minutes)')
        
        # Stress vs heart rate
        if 'stress_avg' in daily_metrics.columns and 'hr_mean' in daily_metrics.columns:
            axes[1, 1].scatter(daily_metrics['stress_avg'], daily_metrics['hr_mean'], alpha=0.7, color='red')
            axes[1, 1].set_title('Stress vs Heart Rate')
            axes[1, 1].set_xlabel('Average Stress Score')
            axes[1, 1].set_ylabel('Average Heart Rate (BPM)')
        
        plt.tight_layout()
        plt.show()
        
        # Overall health score calculation
        print("\nOverall Health Assessment:")
        health_score = 0
        score_components = []
        
        if 'sleep_score' in daily_metrics.columns:
            sleep_component = daily_metrics['sleep_score'].mean()
            score_components.append(('Sleep Quality', sleep_component, 100))
        
        if 'hr_mean' in daily_metrics.columns:
            # Normalize heart rate (60-100 is considered normal)
            hr_score = max(0, min(100, 100 - abs(daily_metrics['hr_mean'].mean() - 70)))
            score_components.append(('Heart Rate', hr_score, 100))
        
        if 'daily_steps' in daily_metrics.columns:
            # Steps score (10,000 steps = 100 points)
            steps_score = min(100, (daily_metrics['daily_steps'].mean() / 10000) * 100)
            score_components.append(('Activity Level', steps_score, 100))
        
        if 'stress_avg' in daily_metrics.columns:
            # Stress score (lower is better)
            stress_score = max(0, 100 - daily_metrics['stress_avg'].mean() * 2)
            score_components.append(('Stress Management', stress_score, 100))
        
        if score_components:
            overall_score = sum(score for _, score, _ in score_components) / len(score_components)
            
            print(f"Overall Health Score: {overall_score:.1f}/100")
            print("\nComponent Scores:")
            for component, score, max_score in score_components:
                print(f"  {component}: {score:.1f}/{max_score}")

else:
    print("Insufficient data for correlation analysis")

Building comprehensive health dashboard...
Insufficient data for correlation analysis


## 8. Health Insights and Recommendations

In [23]:
print("=== HEALTH INSIGHTS AND RECOMMENDATIONS ===")
print()

insights = []
recommendations = []

# Heart rate analysis
if not hr_data.empty and 'heart_rate' in hr_data.columns:
    avg_hr = hr_data['heart_rate'].mean()
    hr_std = hr_data['heart_rate'].std()
    
    insights.append(f"Average resting heart rate: {avg_hr:.1f} BPM (variability: ±{hr_std:.1f})")
    
    if avg_hr < 60:
        insights.append("Heart rate indicates good cardiovascular fitness (bradycardia)")
    elif avg_hr > 100:
        recommendations.append("Consider consulting a healthcare provider about elevated heart rate")
    
    if hr_std > 20:
        insights.append("High heart rate variability detected - may indicate stress or irregular patterns")

# Sleep analysis
if not sleep_data.empty:
    if 'sleep_score' in sleep_data.columns:
        avg_sleep_score = sleep_data['sleep_score'].mean()
        insights.append(f"Average sleep quality score: {avg_sleep_score:.1f}/100")
        
        if avg_sleep_score < 70:
            recommendations.append("Focus on improving sleep quality - consider sleep hygiene practices")
        elif avg_sleep_score > 85:
            insights.append("Excellent sleep quality maintained")
    
    if 'sleep_duration' in sleep_data.columns:
        avg_sleep_duration = sleep_data['sleep_duration'].mean()
        insights.append(f"Average sleep duration: {avg_sleep_duration/60:.1f} hours")
        
        if avg_sleep_duration < 7*60:
            recommendations.append("Consider extending sleep duration - aim for 7-9 hours per night")
        elif avg_sleep_duration > 9*60:
            insights.append("Long sleep duration - monitor for potential sleep disorders")
    
    if 'efficiency' in sleep_data.columns:
        avg_efficiency = sleep_data['efficiency'].mean()
        insights.append(f"Average sleep efficiency: {avg_efficiency:.1f}%")
        
        if avg_efficiency < 85:
            recommendations.append("Improve sleep efficiency - reduce time awake in bed")

# Activity analysis
if not steps_data.empty and 'count' in steps_data.columns:
    daily_steps_avg = steps_data.groupby(steps_data.index.date)['count'].sum().mean()
    insights.append(f"Average daily steps: {daily_steps_avg:.0f}")
    
    if daily_steps_avg < 8000:
        recommendations.append("Increase daily activity - aim for 10,000 steps per day")
    elif daily_steps_avg > 12000:
        insights.append("Excellent activity level maintained")
    
    # Activity consistency
    daily_steps_std = steps_data.groupby(steps_data.index.date)['count'].sum().std()
    if daily_steps_std > 3000:
        insights.append("High variability in daily activity - consider more consistent routine")

# Stress analysis
if not stress_data.empty:
    if 'score' in stress_data.columns:
        avg_stress = stress_data['score'].mean()
        insights.append(f"Average stress level: {avg_stress:.1f}/100")
        
        if avg_stress > 50:
            recommendations.append("Consider stress management techniques - meditation, exercise, adequate sleep")
        elif avg_stress < 30:
            insights.append("Good stress management observed")

# Exercise analysis
if not exercise_data.empty:
    exercise_frequency = len(exercise_data)
    insights.append(f"Exercise sessions recorded: {exercise_frequency} in the analyzed period")
    
    if exercise_frequency < 3 and len(daily_metrics) > 7:  # Less than 3 sessions per week
        recommendations.append("Increase structured exercise frequency - aim for 3-5 sessions per week")
    
    if 'duration' in exercise_data.columns:
        avg_duration = exercise_data['duration'].mean() / 60000  # Convert to minutes
        insights.append(f"Average exercise duration: {avg_duration:.1f} minutes")
        
        if avg_duration < 30:
            recommendations.append("Consider extending exercise duration - aim for 30+ minutes per session")

# Correlation insights
if not daily_metrics.empty and len(daily_metrics) > 3:
    # Check for concerning correlations
    if 'hr_mean' in daily_metrics.columns and 'stress_avg' in daily_metrics.columns:
        hr_stress_corr = daily_metrics[['hr_mean', 'stress_avg']].corr().iloc[0, 1]
        if abs(hr_stress_corr) > 0.6:
            insights.append(f"Strong correlation between heart rate and stress levels (r={hr_stress_corr:.2f})")
    
    if 'sleep_score' in daily_metrics.columns and 'daily_steps' in daily_metrics.columns:
        sleep_activity_corr = daily_metrics[['sleep_score', 'daily_steps']].corr().iloc[0, 1]
        if abs(sleep_activity_corr) > 0.5:
            insights.append(f"Notable correlation between sleep quality and daily activity (r={sleep_activity_corr:.2f})")

# Display insights and recommendations
if insights:
    print("📊 KEY INSIGHTS:")
    for i, insight in enumerate(insights, 1):
        print(f"  {i}. {insight}")
    print()

if recommendations:
    print("💡 RECOMMENDATIONS:")
    for i, rec in enumerate(recommendations, 1):
        print(f"  {i}. {rec}")
    print()

# Health trends summary
print("📈 MONITORING SUMMARY:")
print(f"  • Data collection period: {len(daily_metrics) if not daily_metrics.empty else 'N/A'} days")
print(f"  • Heart rate measurements: {len(hr_data) if not hr_data.empty else 0}")
print(f"  • Sleep sessions: {len(sleep_data) if not sleep_data.empty else 0}")
print(f"  • Activity records: {len(steps_data) if not steps_data.empty else 0}")
print(f"  • Exercise sessions: {len(exercise_data) if not exercise_data.empty else 0}")
print(f"  • Stress measurements: {len(stress_data) if not stress_data.empty else 0}")
print()

print("✅ Analysis complete! Regular monitoring recommended for optimal health insights.")

=== HEALTH INSIGHTS AND RECOMMENDATIONS ===

📈 MONITORING SUMMARY:
  • Data collection period: N/A days
  • Heart rate measurements: 0
  • Sleep sessions: 0
  • Activity records: 0
  • Exercise sessions: 0
  • Stress measurements: 0

✅ Analysis complete! Regular monitoring recommended for optimal health insights.


## 9. Data Export and Reporting

In [24]:
# Export processed data for further analysis
from datetime import datetime

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
export_dir = "health_data_exports"
os.makedirs(export_dir, exist_ok=True)

print(f"Exporting processed data to {export_dir}/...")

# Export daily metrics summary
if not daily_metrics.empty:
    export_file = f"{export_dir}/daily_health_metrics_{timestamp}.csv"
    daily_metrics.to_csv(export_file)
    print(f"✓ Daily metrics exported to {export_file}")

# Export heart rate data
if not hr_data.empty:
    export_file = f"{export_dir}/heart_rate_data_{timestamp}.csv"
    hr_data.to_csv(export_file)
    print(f"✓ Heart rate data exported to {export_file}")

# Export sleep data
if not sleep_data.empty:
    export_file = f"{export_dir}/sleep_data_{timestamp}.csv"
    sleep_data.to_csv(export_file)
    print(f"✓ Sleep data exported to {export_file}")

# Create a summary report
report_file = f"{export_dir}/health_report_{timestamp}.txt"
with open(report_file, 'w') as f:
    f.write("SAMSUNG GALAXY WATCH HEALTH ANALYSIS REPORT\n")
    f.write("="*50 + "\n\n")
    f.write(f"Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
    f.write(f"Device: {primary_device}\n\n")
    
    f.write("DATA SUMMARY:\n")
    f.write(f"  • Analysis period: {len(daily_metrics) if not daily_metrics.empty else 'N/A'} days\n")
    f.write(f"  • Heart rate records: {len(hr_data) if not hr_data.empty else 0}\n")
    f.write(f"  • Sleep sessions: {len(sleep_data) if not sleep_data.empty else 0}\n")
    f.write(f"  • Activity records: {len(steps_data) if not steps_data.empty else 0}\n")
    f.write(f"  • Exercise sessions: {len(exercise_data) if not exercise_data.empty else 0}\n")
    f.write(f"  • Stress measurements: {len(stress_data) if not stress_data.empty else 0}\n\n")
    
    if insights:
        f.write("KEY INSIGHTS:\n")
        for i, insight in enumerate(insights, 1):
            f.write(f"  {i}. {insight}\n")
        f.write("\n")
    
    if recommendations:
        f.write("RECOMMENDATIONS:\n")
        for i, rec in enumerate(recommendations, 1):
            f.write(f"  {i}. {rec}\n")
        f.write("\n")

print(f"✓ Summary report exported to {report_file}")
print(f"\n📁 All exports saved to: {os.path.abspath(export_dir)}")

# Close database connection
client.close()
print("\n✅ Database connection closed.")
print("\n🎉 Analysis complete! Check the exported files for detailed results.")

Exporting processed data to health_data_exports/...
✓ Summary report exported to health_data_exports/health_report_20250716_142049.txt

📁 All exports saved to: /Users/rbastide/ownCloud/ISIS/Recherche/code/galaxy_watch_data/health_data_exports

✅ Database connection closed.

🎉 Analysis complete! Check the exported files for detailed results.
